# 3. Electrograms and the forward-model check

Unipolar signal of each electrode = openCARP's phi_e (referenced to the grounded bath corner),
interpolated at its 8 surface samples and averaged. Only the last beat's window plus twice the filter
history is extracted.

The forward-model check compares openCARP's phi_e with an independent finite-element solve of the same
discrete problem (sigma_m source, same mesh, conductivities and ground) at every electrode sample.
It tests the configuration, not the physiology, and stops the run if it fails.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
result = run.runStage("electrograms")
pd.DataFrame(result["forwardModelCheck"]).T

In [ ]:
egm = np.load(run.out / "electrograms" / "intact.npz")
print({k: egm[k].shape for k in egm.files})

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
for condition in ("intact", "lesion"):
    e = np.load(run.out / "electrograms" / f"{condition}.npz")
    p = int(np.argmin(np.abs(e["placements"][:, 1]) + e["placements"][:, 2]))   # centre y = 0, smallest gap
    ax.plot(e["timeMs"], e["uni"][p, 5], label=f"{condition} E5 (model units, before gain)")
ax.set(xlabel="time (ms)", ylabel="phi_e"); ax.legend(); plt.show()